In [ ]:
#What The Code Does 

#Simulates Rayleigh MIMO channels (K single-antenna users, Nt transmit antennas).

#Computes Zero-Forcing (ZF) beamformers per channel — used as the supervised target (label).

#Trains an MLP (scikit-learn MLPRegressor) to predict the complex beamforming weights from the channel realization (splitting complex into real+imag).

#Evaluates predicted beamformers by computing the sum-rate (bits/s/Hz) and compares ML vs ZF.

#Includes knobs you can tweak (Nt, K, dataset size, model size, training iters).

In [ ]:
#Purpose of the Code : Simulate and optimize beamforming for 5G/B5G/6G networks using a machine learning approach.
#=Instead of solving beamforming analytically, we train a neural network to predict optimal beamforming weights for given channel conditions.

In [ ]:
# Concepts Involved:=

#Channel State Information (CSI): Information about how a signal travels from transmitter to receiver; used to decide beamforming weights.

#Optimal Beamforming Weights: The phase/amplitude settings for antennas that maximize signal quality.

#Singular Value Decomposition (SVD) to find the optimal beamforming vector (similar to maximum ratio transmission).

In [ ]:
#Using Pytorch \ Tensorflow larger models can be trained

In [ ]:
# Beamforming optimization (supervised ML) — Jupyter cell
# Author: ChatGPT
# Requirements: numpy, scikit-learn, matplotlib, pandas
# Paste into a Jupyter cell and run.

import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd

# -------------------------
# User parameters (tweak)
# -------------------------
Nt = 8         # transmit antennas
K = 4          # users (single-antenna each)
SAMPLES = 2000 # number of channel realizations (dataset size)
P_tot = 1.0    # total transmit power
sigma2 = 1e-1  # noise variance
# MLP architecture
MLP_HIDDEN = (256, 128)  # hidden layers sizes
MLP_MAX_ITERS = 150

# -------------------------
# Helper functions
# -------------------------
def complex_to_realvec(z):
    """Convert complex vector -> real-vector [Re... , Im...]"""
    return np.concatenate([np.real(z), np.imag(z)])

def realvec_to_complex(x):
    half = len(x)//2
    return x[:half] + 1j * x[half:]

def compute_zf_beamformers(H, total_power=P_tot):
    """
    H: K x Nt complex channel (each row h_k)
    Returns W: Nt x K complex beamforming matrix (ZF), scaled to total_power equally
    """
    # ZF: W = H^H (H H^H)^-1
    HH = H @ H.conj().T  # K x K
    reg = 1e-6 * np.eye(HH.shape[0])
    inv = np.linalg.inv(HH + reg)
    W = H.conj().T @ inv  # Nt x K
    # normalize columns and scale to equal power per user
    for k in range(W.shape[1]):
        nrm = np.linalg.norm(W[:,k])
        if nrm > 0:
            W[:,k] = W[:,k] / nrm
    W = W * np.sqrt(total_power / W.shape[1])
    return W

def sum_rate(H, W, sigma2):
    """Sum-rate (bits/s/Hz) for given channel H (KxNt) and beamformers W (NtxK)"""
    Kloc = H.shape[0]
    rates = np.zeros(Kloc)
    for k in range(Kloc):
        hk = H[k,:]  # 1 x Nt
        signal = np.abs(hk.dot(W[:,k]))**2
        interf = 0.0
        for j in range(Kloc):
            if j != k:
                interf += np.abs(hk.dot(W[:,j]))**2
        sinr = signal / (interf + sigma2)
        rates[k] = np.log2(1 + sinr)
    return rates.sum()

# -------------------------
# Dataset generation
# -------------------------
X_list = []
Y_list = []
for _ in range(SAMPLES):
    # Rayleigh fading (complex Gaussian)
    H = (np.random.normal(size=(K, Nt)) + 1j*np.random.normal(size=(K, Nt))) / np.sqrt(2.0)
    W_zf = compute_zf_beamformers(H)  # Nt x K
    # flatten H and W and convert to real vectors
    x = complex_to_realvec(H.flatten())
    y = complex_to_realvec(W_zf.flatten())
    X_list.append(x)
    Y_list.append(y)

X = np.array(X_list)  # shape (SAMPLES, 2*Nt*K)
Y = np.array(Y_list)

print("Dataset shapes:", "X:", X.shape, "Y:", Y.shape)

# -------------------------
# Train / test split + scaling
# -------------------------
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
x_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(Y_train)

X_train_s = x_scaler.transform(X_train)
X_test_s = x_scaler.transform(X_test)
Y_train_s = y_scaler.transform(Y_train)

# -------------------------
# Train MLP regressor (supervised)
# -------------------------
mlp = MLPRegressor(hidden_layer_sizes=MLP_HIDDEN,
                   activation='relu',
                   solver='adam',
                   batch_size=64,
                   max_iter=MLP_MAX_ITERS,
                   tol=1e-4,
                   verbose=True,
                   random_state=0)

mlp.fit(X_train_s, Y_train_s)

# -------------------------
# Predict and evaluate
# -------------------------
Y_pred_s = mlp.predict(X_test_s)
Y_pred = y_scaler.inverse_transform(Y_pred_s)

def reshape_W_from_vector(yvec):
    """Turn output vector into Nt x K complex beamformer and re-normalize & scale to meet P_tot"""
    Wc = realvec_to_complex(yvec).reshape(Nt, K)
    for k in range(K):
        nrm = np.linalg.norm(Wc[:,k])
        if nrm > 0:
            Wc[:,k] = Wc[:,k] / nrm
    Wc = Wc * np.sqrt(P_tot / K)
    return Wc

sumrate_zf_list = []
sumrate_pred_list = []
for i in range(X_test.shape[0]):
    # reconstruct H and compute performances
    H_vec = X_test[i]
    Hc = realvec_to_complex(H_vec).reshape(K, Nt)
    W_zf = compute_zf_beamformers(Hc)
    W_pred = reshape_W_from_vector(Y_pred[i])
    sumrate_zf_list.append(sum_rate(Hc, W_zf, sigma2))
    sumrate_pred_list.append(sum_rate(Hc, W_pred, sigma2))

sumrate_zf = np.array(sumrate_zf_list)
sumrate_pred = np.array(sumrate_pred_list)

# -------------------------
# Results / simple plots
# -------------------------
avg_zf = np.mean(sumrate_zf)
avg_pred = np.mean(sumrate_pred)
rel_loss = 100.0 * (avg_zf - avg_pred) / avg_zf if avg_zf != 0 else 0.0

print(f"\nAverage sum-rate (ZF): {avg_zf:.4f}  |  ML-predicted: {avg_pred:.4f}  |  Relative loss: {rel_loss:.2f}%")

# Tabulate summary
metrics = pd.DataFrame({
    "Metric": ["Average Sum-rate (bits/s/Hz)", "Std of Sum-rate"],
    "ZF": [np.mean(sumrate_zf), np.std(sumrate_zf)],
    "ML-predicted": [np.mean(sumrate_pred), np.std(sumrate_pred)],
    "Relative loss (%)": [100*(np.mean(sumrate_zf)-np.mean(sumrate_pred))/np.mean(sumrate_zf) if np.mean(sumrate_zf)!=0 else 0,
                         100*(np.std(sumrate_zf)-np.std(sumrate_pred))/np.std(sumrate_zf) if np.std(sumrate_zf)!=0 else 0]
})
print("\n", metrics)

# Histogram comparison
plt.figure(figsize=(8,4))
plt.hist(sumrate_zf, bins=30, alpha=0.6)
plt.hist(sumrate_pred, bins=30, alpha=0.6)
plt.title("Sum-rate per channel sample: ZF vs ML-predicted")
plt.xlabel("Sum-rate (bits/s/Hz)")
plt.ylabel("Count")
plt.legend(["ZF", "ML-predicted"])
plt.tight_layout()
plt.show()

# Show first few sample comparisons
n_show = min(8, len(sumrate_zf))
df_show = pd.DataFrame({
    "ZF_sumrate": sumrate_zf[:n_show],
    "ML_sumrate": sumrate_pred[:n_show],
    "Ratio (ML/ZF)": sumrate_pred[:n_show] / (sumrate_zf[:n_show] + 1e-12)
})
print("\nFirst few test sample comparisons:")
print(df_show.to_string(index=False))
